## Optimisation Project

### Building and Tuning a Final Model

In [1]:
# 1. Applying All Learning Tuning and Optimisation Techniques.
    # Comprehensive Model Optimisation.
        # Data Preprocessing.
            # Ensure data is clean, scaled, and encoded appropriately.
        # Feature Engineering.
            # Derive new features and select the most important ones.
        # Regularisation.
            # Avoid overfitting by penalising complex models.
        # Cross-Validation.
            # Use techniques like K-Fold or Stratified K-Fold for robust performance metrics.
        # Hyperparameter Tuning.
            # Use methods like GridSearchCV, RandomisedSearchCV, or Bayesian Optimisation.

# 2. Evaluating and Interpreting Model Performance.
    # Performance Metrics.
        # Classification.
            # Accuracy, Precision, Recall, F1 Score, ROC-AUC.
        # Regression.
            # Mean Squared Error (MSE), Mean Absolute Error (MAE), R^2.
        # Importance of Interpretability.
            # Use feature importance and coefficient analysis for transparency.

#### Hands On Project

In [8]:
# Ojective: Build, tune, and optimise a machine learning model using a structured process and evaluste its performance comprehensively.

import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Load dataset.
df = pd.read_csv("Telco-Customer-Churn.csv")

# # Display dataset infor.
# print("Dataset Infor: \n")
# print(df.info())
# print("Class Distribution: \n") # Target 
# print(df["Churn"].value_counts()) # Target (y-axis)
# print("\n Sample Data: \n", df.head())

# Handle missing values.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.fillna({"TotalCharges": df["TotalCharges"].median()}, inplace=True)

# Encode categorical variables.
label_encoder = LabelEncoder()
for column in df.select_dtypes(include=["object"]).columns:
    if column != "Churn":
        df[column] = label_encoder.fit_transform(df[column])

# Encode target variable.
df["Churn"] = label_encoder.fit_transform(df["Churn"])

# Scale numerical features.
scaler = StandardScaler()
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]
numerical_features = scaler.fit_transform(df[numerical_features])

# Features and target.
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Split data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train initial model.
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate and predict intial model.
y_pred_rf = rf_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
cl_rf = classification_report(y_test, y_pred_rf)

print(f"Initial Model Accuracy (Random Forest): {accuracy_rf:.4f}")
print("\n Classification Report (Random Forest): \n", cl_rf)

# Define random parameters.
param_dist = {
    "n_estimators": np.arange(50, 200, 10),
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4]
}

# Initialse Random Search.
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)

# Display best hyperparameter and best score.
print(f"Best Hyperparameters (Random Search): {random_search.best_params_}")
print(f"Best Score (Random Search): {random_search.best_score_:.4f}")

# Train best model.
best_model = random_search.best_estimator_

# Predict and evaluate model.
y_pred_tuned = best_model.predict(X_test)
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
cl_tuned = classification_report(y_test, y_pred_tuned)

print(f"Tuned Model Accuracy: {accuracy_tuned:.4f}")
print("\n Tuned Model Classification Report: \n", cl_tuned)

# Evaluate using cross-validation.
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring="accuracy")

print(f"Cross-Validation Accuracy Scores: {cv_scores}")
print(f"Mean Cross-Validation Accuracy: {cv_scores.mean():.4f}")

Initial Model Accuracy (Random Forest): 0.7949

 Classification Report (Random Forest): 
               precision    recall  f1-score   support

           0       0.83      0.91      0.87      1036
           1       0.65      0.49      0.56       373

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

Best Hyperparameters (Random Search): {'n_estimators': np.int64(60), 'min_samples_split': 20, 'min_samples_leaf': 4, 'max_depth': 15}
Best Score (Random Search): 0.8016
Tuned Model Accuracy: 0.8091

 Tuned Model Classification Report: 
               precision    recall  f1-score   support

           0       0.84      0.91      0.88      1036
           1       0.68      0.52      0.59       373

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409

Cross-Validation Ac